## Trạng thái bản lưu trên GitHub

Bản corrected-angle, nested LOSO và model final đang dùng trên web; xem results/model_06.

Output cell đã được bỏ để tránh file tạm, dữ liệu nhạy cảm và bảng lỗi cũ. Mã cell được giữ; kết quả kiểm chứng nằm trong thư mục `results/`. Xem `docs/notebook-guide.md` trước khi chạy. Không cần chạy tuần tự cả tám notebook.

# 06 — GMDCSA24: góc thân đúng, Nested LOSO và model final

Notebook độc lập, **Run All một lần**. Dùng output tiền xử lý từ notebook 01; không chạy lại MediaPipe. Sửa lỗi kênh thứ sáu: `torso_angle` phải là `global_features[..., 9]`, không phải index 6 (`root_acceleration_x`).

Sau Nested LOSO trên bốn subject, notebook train model final trên đủ 160 video, xuất ONNX và một ZIP gồm cả kết quả đánh giá lẫn model. Đánh giá cross-subject thuộc **model theo fold**, không phải test độc lập của model final. Không dùng cho cảnh báo y tế thực tế.

## 1. Cài đặt
Bật GPU T4. Nếu Kaggle chưa có `onnx`/`onnxruntime`, hãy bật Internet; phụ thuộc được kiểm tra **trước khi train**. Tối đa 60 inner runs, 4 outer runs và một final run; không tắt session giữa chừng.

In [ ]:
from pathlib import Path
import json
import random
import re
import shutil
import time
import hashlib
import importlib.util
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
)
from IPython.display import display

SEED = 42
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 120
PATIENCE = 15
DROPOUT = 0.30

FALL_MULTIPLIERS = [1.0, 1.5, 2.0, 3.0, 4.0]
THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.05), 2)
TARGET_SENSITIVITY = 0.95
MIN_SPECIFICITY = 0.60

CHANNEL_NAMES = [
    "relative_x", "relative_y", "visibility",
    "joint_velocity_x", "joint_velocity_y",
    "torso_angle", "hip_displacement_x", "hip_displacement_y",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd() / "output"
OUTPUT_DIR = WORK_ROOT / "06_corrected_angle_loso"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Fall multipliers:", FALL_MULTIPLIERS)
print("Thresholds:", THRESHOLDS)

def ensure_package(module_name):
    if importlib.util.find_spec(module_name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", module_name], check=True)

ensure_package("onnx")
ensure_package("onnxruntime")
import onnx
import onnxruntime as ort
print("ONNX dependencies: OK")

## 2. Đọc và kiểm tra bốn file
Add Input output của notebook 01 (ưu tiên) hoặc dataset private gồm hai `.zip` NumPy và hai `.txt` CSV. Notebook dừng nếu nhãn, thứ tự video, folds hoặc tên đặc trưng không khớp.

In [ ]:
def find_input_file(*filenames):
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd() / "Kaggle_kiem_tra_2026-09-25", Path.cwd()]
    if not Path("/kaggle").exists():
        roots = roots[2:]
    for filename in filenames:
        matches = sorted({path.resolve() for root in roots if root.exists() for path in root.rglob(filename)})
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(f"Có nhiều file tên {filename}; chỉ gắn một nguồn input: {matches}")
    raise FileNotFoundError(f"Không tìm thấy một trong các file: {filenames}. Hãy Add Input output notebook 01 hoặc dataset private gồm bốn file.")

FEATURE_PATH = find_input_file("gmdcsa24_features_32.npz", "gmdcsa24_features_32.zip")
POSE_PATH = find_input_file("gmdcsa24_pose_32.npz", "gmdcsa24_pose_32.zip")
MANIFEST_PATH = find_input_file("gmdcsa24_pose_manifest.csv", "pose_mnf.txt")
FOLDS_PATH = find_input_file("gmdcsa24_cross_subject_folds.csv", "cross.txt")

with np.load(FEATURE_PATH, allow_pickle=False) as saved:
    feature_data = {name: saved[name] for name in saved.files}
with np.load(POSE_PATH, allow_pickle=False) as saved:
    pose_data = {name: saved[name] for name in saved.files}
manifest = pd.read_csv(MANIFEST_PATH)
saved_folds = pd.read_csv(FOLDS_PATH)

required_features = {"skeleton_features", "motion_features", "global_features", "global_feature_names", "video_ids", "labels", "subjects", "durations"}
assert required_features.issubset(feature_data), required_features - set(feature_data)
assert {"poses", "video_ids", "labels", "subjects"}.issubset(pose_data)
names = feature_data["global_feature_names"].astype(str).tolist()
assert len(names) == len(set(names)) == 13, names
assert names[6] == "root_acceleration_x" and names[9] == "torso_angle", names
assert names[:2] == ["root_x", "root_y"], names

video_ids = feature_data["video_ids"].astype(str)
y = feature_data["labels"].astype(np.int64)
subjects = feature_data["subjects"].astype(np.int64)
assert len(video_ids) == len(set(video_ids)) == 160
assert np.array_equal(np.unique(y), [0, 1])
assert np.array_equal(np.unique(subjects), [1, 2, 3, 4])
assert np.array_equal(pose_data["video_ids"].astype(str), video_ids)
assert np.array_equal(pose_data["labels"], y)
assert np.array_equal(pose_data["subjects"], subjects)
assert np.array_equal(manifest["video_id"].astype(str).to_numpy(), video_ids)
assert np.array_equal(manifest["label_id"].to_numpy(), y)
assert np.array_equal(manifest["subject"].to_numpy(), subjects)
assert len(saved_folds) == 4 * len(y)
assert saved_folds["fold"].nunique() == 4
assert saved_folds.groupby("fold")["sample_index"].nunique().eq(len(y)).all()
indices = saved_folds["sample_index"].to_numpy(dtype=np.int64)
assert np.array_equal(saved_folds["video_id"].astype(str).to_numpy(), video_ids[indices])
assert np.array_equal(saved_folds["label_id"].to_numpy(), y[indices])
assert np.array_equal(saved_folds["subject"].to_numpy(), subjects[indices])

skeleton = feature_data["skeleton_features"].astype(np.float32)
motion = feature_data["motion_features"].astype(np.float32)
global_features = feature_data["global_features"].astype(np.float32)
assert skeleton.shape == (160, 32, 33, 4)
assert motion.shape == (160, 32, 33, 6)
assert global_features.shape == (160, 32, 13)
assert pose_data["poses"].shape == (160, 32, 33, 4)
assert np.all(feature_data["durations"] > 0)

joint_channels = np.stack([
    skeleton[..., 0], skeleton[..., 1], skeleton[..., 3],
    motion[..., 0], motion[..., 1],
], axis=-1)
hip_position = global_features[..., [names.index("root_x"), names.index("root_y")]]
hip_displacement = hip_position - hip_position[:, :1, :]
torso_index = names.index("torso_angle")
torso_angle = global_features[..., torso_index:torso_index + 1]
body_channels = np.concatenate([torso_angle, hip_displacement], axis=-1)
body_channels = np.repeat(body_channels[:, :, None, :], 33, axis=2)
X = np.transpose(np.concatenate([joint_channels, body_channels], axis=-1), (0, 3, 1, 2)).astype(np.float32)
assert X.shape == (160, 8, 32, 33) and np.isfinite(X).all()
assert np.array_equal(X[:, 5, :, 0], global_features[..., 9])
assert np.array_equal(X[:, 6, :, 0], hip_displacement[..., 0])
assert np.array_equal(X[:, 7, :, 0], hip_displacement[..., 1])

print("Input OK:", FEATURE_PATH, POSE_PATH, MANIFEST_PATH, FOLDS_PATH, sep="\n- ")
print("Samples:", len(y), "ADL:", int((y == 0).sum()), "FALL:", int((y == 1).sum()))
print("X:", X.shape, "| torso_angle index:", torso_index, "| finite:", np.isfinite(X).all())

## 3. Graph, augmentation và kiến trúc
Giữ kiến trúc 04b, augmentation chỉ dùng cho train. Thử ONNX chưa train trước thí nghiệm dài để phát hiện sớm lỗi môi trường.

In [ ]:
POSE_EDGES = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
    (27,29),(28,30),(29,31),(30,32),(27,31),(28,32),
]

LEFT_RIGHT_PAIRS = [
    (1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
    (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32),
]

def make_normalized_adjacency(num_nodes=33):
    adjacency = np.eye(num_nodes, dtype=np.float32)
    for source, target in POSE_EDGES:
        adjacency[source, target] = 1.0
        adjacency[target, source] = 1.0
    degree = adjacency.sum(axis=1)
    inv_sqrt = np.power(np.maximum(degree, 1e-8), -0.5)
    adjacency = inv_sqrt[:, None] * adjacency * inv_sqrt[None, :]
    return torch.tensor(adjacency, dtype=torch.float32)

ADJACENCY = make_normalized_adjacency()

def augment_sequence(sequence):
    sequence = sequence.clone()
    _, num_frames, _ = sequence.shape

    # Crop 80–100% thời gian rồi nội suy lại 32 frame.
    if random.random() < 0.50:
        crop_length = random.randint(max(8, int(0.80 * num_frames)), num_frames)
        start = random.randint(0, num_frames - crop_length)
        crop = sequence[:, start:start + crop_length, :]
        sequence = F.interpolate(
            crop.permute(2, 0, 1),
            size=num_frames,
            mode="linear",
            align_corners=False,
        ).permute(1, 2, 0)

    # Lật ngang và hoán đổi khớp trái-phải.
    if random.random() < 0.50:
        original = sequence.clone()
        for left, right in LEFT_RIGHT_PAIRS:
            sequence[:, :, left] = original[:, :, right]
            sequence[:, :, right] = original[:, :, left]
        sequence[[0, 3, 5, 6]] *= -1.0

    # Nhiễu nhẹ trên tọa độ x, y.
    if random.random() < 0.35:
        sequence[0:2] += torch.randn_like(sequence[0:2]) * 0.01

    # Bỏ ngẫu nhiên 1–3 joint.
    if random.random() < 0.20:
        joint_ids = torch.randperm(sequence.shape[2])[:random.randint(1, 3)]
        sequence[:, :, joint_ids] = 0.0

    return sequence

class PoseSequenceDataset(Dataset):
    def __init__(self, features, labels, augment=False):
        self.features = torch.from_numpy(features).float()
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        sequence = self.features[index]
        if self.augment:
            sequence = augment_sequence(sequence)
        return sequence, self.labels[index]

def make_loader(indices, augment, shuffle, seed):
    dataset = PoseSequenceDataset(X[indices], y[indices], augment=augment)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )

print("Adjacency:", tuple(ADJACENCY.shape))
print("Số cạnh:", len(POSE_EDGES))

In [ ]:
class SpatialGraphConv(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency):
        super().__init__()
        self.projection = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.register_buffer("adjacency", adjacency.clone())
        self.edge_importance = nn.Parameter(torch.ones_like(adjacency))

    def forward(self, x):
        x = self.projection(x)
        graph = self.adjacency * self.edge_importance
        return torch.einsum("nctv,vw->nctw", x, graph)

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, adjacency, stride=1, dropout=0.0):
        super().__init__()
        self.graph_conv = SpatialGraphConv(in_channels, out_channels, adjacency)
        self.temporal_conv = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0),
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout),
        )
        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels),
            )
        self.activation = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.activation(self.temporal_conv(self.graph_conv(x)) + self.residual(x))

class MESTGCNTA(nn.Module):
    def __init__(self, adjacency, in_channels=8, num_classes=2, dropout=DROPOUT):
        super().__init__()
        num_nodes = adjacency.shape[0]
        self.data_bn = nn.BatchNorm1d(in_channels * num_nodes)
        self.blocks = nn.Sequential(
            STGCNBlock(in_channels, 64, adjacency, stride=1, dropout=dropout),
            STGCNBlock(64, 64, adjacency, stride=1, dropout=dropout),
            STGCNBlock(64, 128, adjacency, stride=2, dropout=dropout),
            STGCNBlock(128, 128, adjacency, stride=1, dropout=dropout),
        )
        self.temporal_attention = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=1),
            nn.Tanh(),
            nn.Conv1d(64, 1, kernel_size=1),
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        batch_size, channels, frames, nodes = x.shape
        x = x.permute(0, 3, 1, 2).contiguous().view(batch_size, nodes * channels, frames)
        x = self.data_bn(x)
        x = x.view(batch_size, nodes, channels, frames).permute(0, 2, 3, 1).contiguous()
        x = self.blocks(x)
        temporal_features = x.mean(dim=3)
        attention = torch.softmax(self.temporal_attention(temporal_features), dim=2)
        video_features = (temporal_features * attention).sum(dim=2)
        logits = self.classifier(video_features)
        return logits, attention.squeeze(1)

model_check = MESTGCNTA(ADJACENCY).to(DEVICE)
with torch.no_grad():
    logits_check, attention_check = model_check(torch.zeros(2, 8, 32, 33, device=DEVICE))
print("Logits:", tuple(logits_check.shape))
print("Attention:", tuple(attention_check.shape))
print("Số tham số:", f"{sum(p.numel() for p in model_check.parameters()):,}")
del model_check
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import io
class DeploymentModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pose_features):
        logits, _ = self.model(pose_features)
        return torch.softmax(logits, dim=1)[:, 1]

probe = DeploymentModel(MESTGCNTA(ADJACENCY).to(DEVICE)).eval()
buffer = io.BytesIO()
torch.onnx.export(
    probe, torch.from_numpy(X[:1]).to(DEVICE), buffer, export_params=True,
    opset_version=17, do_constant_folding=True,
    input_names=["pose_features"], output_names=["fall_probability"],
    dynamic_axes={"pose_features": {0: "batch_size"}, "fall_probability": {0: "batch_size"}},
    dynamo=False,
)
onnx.checker.check_model(onnx.load_model_from_string(buffer.getvalue()))
runtime = ort.InferenceSession(buffer.getvalue(), providers=["CPUExecutionProvider"])
probe_out = runtime.run(["fall_probability"], {"pose_features": X[:1]})[0]
assert probe_out.shape == (1,) and np.isfinite(probe_out).all()
print("Preflight model + ONNX export + ONNX Runtime: PASS")
del probe, runtime, buffer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Metric và hàm train
Outer test giữ riêng một subject; inner validation chỉ dùng ba subject còn lại để chọn hệ số Fall, epoch và ngưỡng. Mục tiêu sensitivity 0.95 là **tiêu chí chọn**, không phải cam kết đạt được.

In [ ]:
def metrics_from_predictions(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "f2": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "false_alarm_rate": 1.0 - specificity,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def metrics_at_threshold(y_true, probabilities, threshold):
    result = metrics_from_predictions(y_true, (probabilities >= threshold).astype(np.int64))
    result["threshold"] = float(threshold)
    return result

def operating_rank(metrics):
    sensitivity = metrics["sensitivity"]
    specificity = metrics["specificity"]
    f2 = metrics["f2"]
    if sensitivity >= TARGET_SENSITIVITY and specificity >= MIN_SPECIFICITY:
        return (2, specificity, f2, sensitivity)
    if sensitivity >= TARGET_SENSITIVITY:
        return (1, specificity, f2, sensitivity)
    return (0, sensitivity, specificity, f2)

def choose_threshold(y_true, probabilities):
    candidates = []
    for threshold in THRESHOLDS:
        result = metrics_at_threshold(y_true, probabilities, threshold)
        result["rank"] = operating_rank(result)
        candidates.append(result)
    return max(candidates, key=lambda item: item["rank"])

print("Quy tắc threshold: OK")

In [ ]:
def make_class_weights(train_indices, fall_multiplier):
    counts = np.bincount(y[train_indices], minlength=2).astype(np.float32)
    if np.any(counts == 0):
        raise ValueError(f"Train split thiếu lớp: {counts}")
    weights = len(train_indices) / (2.0 * counts)
    weights[1] *= fall_multiplier
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    total_items = 0
    for features, labels in loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_items += len(labels)
    return total_loss / max(total_items, 1)

@torch.no_grad()
def predict_probabilities(model, loader):
    model.eval()
    labels_all, probabilities_all, attention_all = [], [], []
    for features, labels in loader:
        logits, attention = model(features.to(DEVICE, non_blocking=True))
        probabilities = torch.softmax(logits, dim=1)[:, 1]
        labels_all.append(labels.numpy())
        probabilities_all.append(probabilities.cpu().numpy())
        attention_all.append(attention.cpu().numpy())
    return (
        np.concatenate(labels_all),
        np.concatenate(probabilities_all),
        np.concatenate(attention_all),
    )

def fit_inner_fold(train_indices, val_indices, fall_multiplier, run_seed):
    set_seed(run_seed)
    train_loader = make_loader(train_indices, True, True, run_seed)
    val_loader = make_loader(val_indices, False, False, run_seed)
    model = MESTGCNTA(ADJACENCY).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=make_class_weights(train_indices, fall_multiplier))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_state = None
    best_epoch = 0
    best_choice = None
    best_probabilities = None
    stale_epochs = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        train_one_epoch(model, train_loader, criterion, optimizer)
        val_labels, val_probabilities, _ = predict_probabilities(model, val_loader)
        choice = choose_threshold(val_labels, val_probabilities)

        if best_choice is None or choice["rank"] > best_choice["rank"]:
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            best_epoch = epoch
            best_choice = choice
            best_probabilities = val_probabilities.copy()
            stale_epochs = 0
        else:
            stale_epochs += 1

        if stale_epochs >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_choice": best_choice,
        "val_probabilities": best_probabilities,
        "epochs_ran": epoch,
    }

def fit_fixed_epochs(train_indices, fall_multiplier, num_epochs, run_seed):
    set_seed(run_seed)
    loader = make_loader(train_indices, True, True, run_seed)
    model = MESTGCNTA(ADJACENCY).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=make_class_weights(train_indices, fall_multiplier))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    for _ in range(num_epochs):
        train_one_epoch(model, loader, criterion, optimizer)
    return model

print("Hàm train: OK")

In [ ]:
def run_nested_loso_sensitivity():
    unique_subjects = sorted(np.unique(subjects).tolist())
    num_samples = len(y)
    oof_probabilities = np.full(num_samples, np.nan, dtype=np.float32)
    oof_predictions = np.full(num_samples, -1, dtype=np.int64)
    oof_thresholds = np.full(num_samples, np.nan, dtype=np.float32)
    oof_multipliers = np.full(num_samples, np.nan, dtype=np.float32)
    outer_records, inner_records, search_records = [], [], []
    start_time = time.time()

    for outer_number, test_subject in enumerate(unique_subjects, start=1):
        print("\n" + "=" * 76)
        print(f"OUTER {outer_number}/{len(unique_subjects)} | Test Subject {test_subject}")
        test_indices = np.where(subjects == test_subject)[0]
        outer_train_indices = np.where(subjects != test_subject)[0]
        inner_subjects = [subject for subject in unique_subjects if subject != test_subject]
        best_candidate = None

        for multiplier_number, fall_multiplier in enumerate(FALL_MULTIPLIERS):
            pooled_labels, pooled_probabilities, best_epochs = [], [], []
            print(f"  Multiplier {fall_multiplier:.1f}", end="")

            for inner_number, val_subject in enumerate(inner_subjects):
                val_indices = np.where(subjects == val_subject)[0]
                train_indices = np.where((subjects != test_subject) & (subjects != val_subject))[0]
                run_seed = SEED + outer_number * 10000 + multiplier_number * 1000 + inner_number * 100
                result = fit_inner_fold(train_indices, val_indices, fall_multiplier, run_seed)

                pooled_labels.append(y[val_indices])
                pooled_probabilities.append(result["val_probabilities"])
                best_epochs.append(result["best_epoch"])
                metric = result["best_choice"]
                inner_records.append({
                    "outer_test_subject": test_subject,
                    "inner_val_subject": val_subject,
                    "fall_multiplier": fall_multiplier,
                    "best_epoch": result["best_epoch"],
                    "epochs_ran": result["epochs_ran"],
                    "threshold_at_best_epoch": metric["threshold"],
                    "sensitivity": metric["sensitivity"],
                    "specificity": metric["specificity"],
                    "f2": metric["f2"],
                })
                del result["model"]
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            pooled_labels = np.concatenate(pooled_labels)
            pooled_probabilities = np.concatenate(pooled_probabilities)
            pooled_choice = choose_threshold(pooled_labels, pooled_probabilities)
            final_epochs = max(1, int(np.median(best_epochs)))
            candidate = {
                "fall_multiplier": fall_multiplier,
                "threshold": pooled_choice["threshold"],
                "metrics": pooled_choice,
                "rank": pooled_choice["rank"],
                "best_epochs": best_epochs,
                "final_epochs": final_epochs,
            }
            search_records.append({
                "outer_test_subject": test_subject,
                "fall_multiplier": fall_multiplier,
                "selected_threshold": pooled_choice["threshold"],
                "median_epoch": final_epochs,
                "inner_sensitivity": pooled_choice["sensitivity"],
                "inner_specificity": pooled_choice["specificity"],
                "inner_f2": pooled_choice["f2"],
                "target_reached": pooled_choice["sensitivity"] >= TARGET_SENSITIVITY,
            })
            if best_candidate is None or candidate["rank"] > best_candidate["rank"]:
                best_candidate = candidate

            print(
                f" | Sens={pooled_choice['sensitivity']:.3f}"
                f" Spec={pooled_choice['specificity']:.3f}"
                f" F2={pooled_choice['f2']:.3f}"
                f" threshold={pooled_choice['threshold']:.2f}"
                f" epochs={best_epochs}"
            )

        selected_multiplier = best_candidate["fall_multiplier"]
        selected_threshold = best_candidate["threshold"]
        final_epochs = best_candidate["final_epochs"]
        print(
            f"  -> Chọn multiplier={selected_multiplier:.1f}, "
            f"threshold={selected_threshold:.2f}, epochs={final_epochs}"
        )

        final_seed = SEED + outer_number * 10000 + 9999
        final_model = fit_fixed_epochs(
            outer_train_indices, selected_multiplier, final_epochs, final_seed
        )
        test_loader = make_loader(test_indices, False, False, final_seed)
        test_labels, test_probabilities, test_attention = predict_probabilities(final_model, test_loader)
        test_predictions = (test_probabilities >= selected_threshold).astype(np.int64)
        test_metrics = metrics_from_predictions(test_labels, test_predictions)

        oof_probabilities[test_indices] = test_probabilities
        oof_predictions[test_indices] = test_predictions
        oof_thresholds[test_indices] = selected_threshold
        oof_multipliers[test_indices] = selected_multiplier
        outer_records.append({
            "outer_test_subject": test_subject,
            "fall_multiplier": selected_multiplier,
            "threshold": selected_threshold,
            "final_epochs": final_epochs,
            **test_metrics,
        })

        checkpoint = {
            "model_state_dict": {name: value.detach().cpu() for name, value in final_model.state_dict().items()},
            "adjacency": ADJACENCY.cpu(),
            "channel_names": CHANNEL_NAMES,
            "sequence_length": 32,
            "num_joints": 33,
            "label_mapping": {"ADL": 0, "FALL": 1},
            "outer_test_subject": test_subject,
            "trained_subjects": inner_subjects,
            "fall_multiplier": selected_multiplier,
            "decision_threshold": selected_threshold,
            "epochs": final_epochs,
            "seed": final_seed,
        }
        checkpoint_path = MODEL_DIR / f"outer_test_subject_{test_subject}.pt"
        torch.save(checkpoint, checkpoint_path)

        print(
            f"  Test S{test_subject}: Sens={test_metrics['sensitivity']:.4f} | "
            f"Spec={test_metrics['specificity']:.4f} | "
            f"FN={test_metrics['fn']} | FP={test_metrics['fp']}"
        )
        pd.DataFrame(outer_records).to_csv(OUTPUT_DIR / "outer_partial.csv", index=False)
        pd.DataFrame(inner_records).to_csv(OUTPUT_DIR / "inner_partial.csv", index=False)
        pd.DataFrame(search_records).to_csv(OUTPUT_DIR / "search_partial.csv", index=False)

        del final_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        "oof_probabilities": oof_probabilities,
        "oof_predictions": oof_predictions,
        "oof_thresholds": oof_thresholds,
        "oof_multipliers": oof_multipliers,
        "outer_records": outer_records,
        "inner_records": inner_records,
        "search_records": search_records,
        "elapsed_seconds": time.time() - start_time,
    }

print("Nested LOSO: sẵn sàng")

## 5. Chạy Nested LOSO
Cell lâu nhất. Checkpoint và bảng tạm của từng outer fold được lưu để nếu phiên dừng có thể xem đã đi đến đâu.

In [ ]:
experiment = run_nested_loso_sensitivity()
print("\nHoàn thành sau:", round(experiment["elapsed_seconds"] / 60, 2), "phút")

## 6. Báo cáo cross-subject
Không dùng kết quả outer test để chọn lại tham số. Không chép các metric cũ của 04b vào kết quả mới.

In [ ]:
oof_probabilities = experiment["oof_probabilities"]
oof_predictions = experiment["oof_predictions"]
oof_thresholds = experiment["oof_thresholds"]
outer_df = pd.DataFrame(experiment["outer_records"])
inner_df = pd.DataFrame(experiment["inner_records"])
search_df = pd.DataFrame(experiment["search_records"])
assert len(outer_df) == 4 and len(inner_df) == 4 * 3 * len(FALL_MULTIPLIERS)
assert np.isfinite(oof_probabilities).all() and np.all(oof_predictions >= 0)
assert np.isfinite(oof_thresholds).all()

pooled_metrics = metrics_from_predictions(y, oof_predictions)
prediction_df = pd.DataFrame({
    "sample_index": np.arange(len(y)), "video_id": video_ids,
    "subject": subjects, "label": y,
    "fall_probability": oof_probabilities,
    "threshold": oof_thresholds,
    "prediction": oof_predictions,
})
prediction_df["correct"] = prediction_df["label"] == prediction_df["prediction"]
outer_df.to_csv(OUTPUT_DIR / "outer_fold_results.csv", index=False)
inner_df.to_csv(OUTPUT_DIR / "inner_fold_results.csv", index=False)
search_df.to_csv(OUTPUT_DIR / "inner_search_summary.csv", index=False)
prediction_df.to_csv(OUTPUT_DIR / "outer_test_predictions_160.csv", index=False)
summary = {
    "protocol": "Nested LOSO; corrected torso_angle index 9",
    "samples": int(len(y)), "subjects": sorted(np.unique(subjects).tolist()),
    "target_sensitivity": TARGET_SENSITIVITY,
    "pooled_metrics": {name: float(value) if isinstance(value, (float, np.floating)) else int(value) for name, value in pooled_metrics.items()},
    "note": "Outer-test predictions evaluate fold models, not the final all-subject model.",
}
(OUTPUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Nested LOSO | ACC:", round(pooled_metrics["accuracy"], 4),
      "SENS:", round(pooled_metrics["sensitivity"], 4),
      "SPEC:", round(pooled_metrics["specificity"], 4),
      "FN:", pooled_metrics["fn"], "FP:", pooled_metrics["fp"])
print(outer_df[["outer_test_subject", "fall_multiplier", "threshold", "final_epochs", "sensitivity", "specificity", "fn", "fp"]].to_string(index=False))
print("Sensitivity >= 0.95:", pooled_metrics["sensitivity"] >= TARGET_SENSITIVITY)

## 7. Train final trên đủ 160 video
Tham số lấy trung vị từ các cấu hình đã chọn trong vòng trong; ngưỡng final là heuristic, không phải một test độc lập.

In [ ]:
EVAL_DIR = OUTPUT_DIR
OUTPUT_DIR = EVAL_DIR / "deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FALL_MULTIPLIER = float(np.median(outer_df["fall_multiplier"].to_numpy()))
FINAL_EPOCHS = max(1, int(np.median(outer_df["final_epochs"].to_numpy())))
DECISION_THRESHOLD = float(np.median(outer_df["threshold"].to_numpy()))
FINAL_SEED = SEED + 50000
print("Heuristic from inner-selected fold settings:", FALL_MULTIPLIER, FINAL_EPOCHS, DECISION_THRESHOLD)
print("This threshold has NOT been independently validated for the final model.")
SEED = FINAL_SEED
set_seed(SEED)

In [ ]:
all_indices = np.arange(len(y))
class_counts = np.bincount(y, minlength=2).astype(np.float32)
class_weights = len(y) / (2.0 * class_counts)
class_weights[1] *= FALL_MULTIPLIER
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

train_loader = make_loader(
    all_indices,
    augment=True,
    shuffle=True,
    seed=SEED,
)

final_model = MESTGCNTA(ADJACENCY).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = torch.optim.AdamW(
    final_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = []
start_time = time.time()

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()
    total_loss = 0.0
    total_items = 0

    for features, labels in train_loader:
        features = features.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = final_model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_items += len(labels)

    epoch_loss = total_loss / total_items
    history.append({"epoch": epoch, "train_loss": epoch_loss})
    print(f"Epoch {epoch:02d}/{FINAL_EPOCHS} | train_loss={epoch_loss:.6f}")

elapsed_seconds = time.time() - start_time
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

print("\nHoàn thành trong:", round(elapsed_seconds, 2), "giây")
print("Class counts [ADL, FALL]:", class_counts.astype(int).tolist())
print("Class weights:", class_weights.tolist())

## 8. Checkpoint và ONNX
Kiểm tra output hữu hạn, kiểm tra cấu trúc ONNX và so sánh ONNX–PyTorch; dừng nếu sai số vượt `1e-4`.

In [ ]:
final_model.eval()
with torch.no_grad():
    sample_tensor = torch.from_numpy(X[:4]).float().to(DEVICE)
    sample_logits, sample_attention = final_model(sample_tensor)
    sample_probabilities = torch.softmax(sample_logits, dim=1)[:, 1]

assert torch.isfinite(sample_logits).all()
assert torch.isfinite(sample_attention).all()
assert torch.isfinite(sample_probabilities).all()

checkpoint_path = OUTPUT_DIR / "me_stgcn_ta_corrected_all_subjects.pt"
checkpoint = {
    "model_state_dict": {
        name: value.detach().cpu()
        for name, value in final_model.state_dict().items()
    },
    "architecture": "ME-STGCN-TA",
    "adjacency": ADJACENCY.cpu(),
    "input_shape": [None, 8, 32, 33],
    "channel_names": CHANNEL_NAMES,
    "sequence_length": 32,
    "num_joints": 33,
    "label_mapping": {"ADL": 0, "FALL": 1},
    "fall_multiplier": FALL_MULTIPLIER,
    "decision_threshold": DECISION_THRESHOLD,
    "epochs": FINAL_EPOCHS,
    "seed": SEED,
    "training_samples": int(len(y)),
    "training_subjects": sorted(np.unique(subjects).tolist()),
    "torso_angle_global_index": torso_index,
    "source_evaluation": summary,
}
torch.save(checkpoint, checkpoint_path)
np.save(OUTPUT_DIR / "mediapipe_adjacency_33.npy", ADJACENCY.numpy())
np.save(OUTPUT_DIR / "sample_input.npy", X[:1])

print("Sample probabilities:", sample_probabilities.cpu().numpy())
print("Attention shape:", tuple(sample_attention.shape))
print("Đã lưu:", checkpoint_path)

In [ ]:
def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package_name = pip_name or import_name
        print(f"Đang cài {package_name}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package_name
        ])

ensure_package("onnx")
ensure_package("onnxruntime")

import onnx
import onnxruntime as ort

class DeploymentModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pose_features):
        logits, _ = self.model(pose_features)
        return torch.softmax(logits, dim=1)[:, 1]

deployment_model = DeploymentModel(final_model).to(DEVICE).eval()
dummy_input = torch.zeros(1, 8, 32, 33, dtype=torch.float32, device=DEVICE)
onnx_path = OUTPUT_DIR / "me_stgcn_ta_corrected_all_subjects.onnx"

torch.onnx.export(
    deployment_model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["pose_features"],
    output_names=["fall_probability"],
    dynamic_axes={
        "pose_features": {0: "batch_size"},
        "fall_probability": {0: "batch_size"},
    },
    dynamo=False,
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print("ONNX hợp lệ:", onnx_path)
print("Dung lượng:", round(onnx_path.stat().st_size / 1024 / 1024, 2), "MB")

In [ ]:
verification_input = X[:8].astype(np.float32)
with torch.no_grad():
    pytorch_output = deployment_model(
        torch.from_numpy(verification_input).to(DEVICE)
    ).cpu().numpy()

session = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"],
)
onnx_output = session.run(
    ["fall_probability"],
    {"pose_features": verification_input},
)[0]

max_absolute_error = float(np.max(np.abs(pytorch_output - onnx_output)))
print("PyTorch:", pytorch_output)
print("ONNX:", onnx_output)
print("Max absolute error:", max_absolute_error)

if max_absolute_error > 1e-4:
    raise RuntimeError("ONNX không khớp PyTorch; chưa được triển khai web")
print("PyTorch–ONNX parity: PASS")

## 9. Cấu hình và gói tải về
Tải `gmdcsa24_corrected_angle_complete.zip` ở Output khi Save & Run All hoàn tất. Giữ ZIP cũ làm đối chứng. Webcam realtime là bước kiểm chứng tiếp theo, không được suy từ kết quả clip 32 frame.

In [ ]:
deployment_config = {
    "model_name": "ME-STGCN-TA corrected torso angle, final all subjects",
    "model_file": "me_stgcn_ta_corrected_all_subjects.onnx",
    "pytorch_file": "me_stgcn_ta_corrected_all_subjects.pt",
    "input_name": "pose_features",
    "output_name": "fall_probability",
    "input_layout": "NCTV",
    "input_shape": [None, 8, 32, 33],
    "dtype": "float32",
    "channel_names": CHANNEL_NAMES,
    "sequence_length": 32,
    "num_joints": 33,
    "decision_threshold": DECISION_THRESHOLD,
    "decision_threshold_note": "Median inner-selected outer-fold thresholds; final model has no independent test.",
    "torso_angle_global_index": torso_index,
    "label_mapping": {"0": "ADL", "1": "FALL"},
    "feature_definition": {
        "relative_xy": "hip-centered and divided by per-video torso scale",
        "visibility": "MediaPipe visibility",
        "joint_velocity_xy": "difference of normalized joint coordinates divided by duration_seconds / 31; first frame zero",
        "torso_angle": "global_features index 9, repeated over 33 joints",
        "hip_displacement_xy": "raw normalized-image hip position minus first-frame hip position, repeated over 33 joints",
    },
    "training": {
        "samples": int(len(y)),
        "subjects": sorted(np.unique(subjects).tolist()),
        "epochs": FINAL_EPOCHS,
        "fall_multiplier": FALL_MULTIPLIER,
        "seed": SEED,
    },
    "evaluation_reference": summary,
    "realtime_warning": (
        "Training clips were resampled as complete clips to 32 frames. "
        "A fixed realtime window length must be validated before claiming realtime performance."
    ),
}

config_path = OUTPUT_DIR / "model_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(deployment_config, file, indent=2, ensure_ascii=False)

readme_text = f"""GMDCSA24 ME-STGCN-TA deployment bundle

Files
- me_stgcn_ta_corrected_all_subjects.pt: PyTorch checkpoint
- me_stgcn_ta_corrected_all_subjects.onnx: browser/backend inference model
- model_config.json: input channels, threshold and metadata
- mediapipe_adjacency_33.npy: graph adjacency
- sample_input.npy: one valid input tensor with shape (1,8,32,33)
- training_history.csv: final training loss

Important
- Decision threshold: {DECISION_THRESHOLD}
- The model input is not raw RGB video.
- MediaPipe preprocessing must exactly reproduce the 8 channels.
- Nested LOSO in notebook 06 evaluates four fold models; the final model has no independent test set.
- The final threshold is a median heuristic and needs testing on new videos.
- Realtime temporal-window duration still requires validation.
"""
(OUTPUT_DIR / "README.txt").write_text(readme_text, encoding="utf-8")

print("Config:", config_path)
provenance = {
    "feature_sha256": hashlib.sha256(FEATURE_PATH.read_bytes()).hexdigest(),
    "pose_sha256": hashlib.sha256(POSE_PATH.read_bytes()).hexdigest(),
    "global_feature_names": names,
    "corrected_torso_angle_index": torso_index,
    "onnx_pytorch_max_error": max_absolute_error,
}
(EVAL_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
bundle_path = shutil.make_archive(str(WORK_ROOT / "gmdcsa24_corrected_angle_complete"), "zip", EVAL_DIR)
with zipfile.ZipFile(bundle_path) as checked:
    assert checked.testzip() is None
    assert "deployment/me_stgcn_ta_corrected_all_subjects.onnx" in checked.namelist()
    assert "summary.json" in checked.namelist()
print("Tải ZIP duy nhất:", bundle_path)